# Generate RAG Evaluation Dataset

This notebook has two responsibilities:
1. Re-index the regulatory corpus from scratch.
2. Generate the RAG evaluation dataset and save it to `eval/dataset.json`.

In [ ]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv()

from tools import gcs_tools
from rag.indexer import index_documents
from rag import dense, sparse
from tools.llm_tools import get_llm

## Step 1 — Re-index the corpus

In [ ]:
from tools import postgresql_tools, elasticsearch_tools, falkordb_tools

print("Clearing existing indexes...")
postgresql_tools.clear_table()
elasticsearch_tools.clear_index()
falkordb_tools.clear_graph()
print("Indexes cleared.")

print("Re-indexing corpus...")
index_documents(
    before = lambda doc: print('Starting with %s' % doc['title']),
    after  = lambda doc: print('Finished with %s' % doc['title']),
)

n_docs = len(gcs_tools.get_document_catalog())
print(f"Indexing complete. {n_docs} document(s) indexed.")

## Step 2 — Build per-country corpus

In [ ]:
from langchain_core.documents import Document
from collections import defaultdict

catalog = gcs_tools.get_document_catalog()

corpus_by_country: dict[str, list[Document]] = defaultdict(list)

for doc in catalog:
    document_id = doc["document_id"]
    country_code = doc["country_code"]
    content = gcs_tools.get_document(document_id)
    lc_doc = Document(page_content=content, metadata=doc)
    corpus_by_country[country_code].append(lc_doc)

print("Documents per country:")
for country_code, docs in sorted(corpus_by_country.items()):
    print(f"  {country_code}: {len(docs)} document(s)")

## Step 3 — Generate evaluation dataset

In [ ]:
import os
from openai import OpenAI
from ragas.testset import TestsetGenerator
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.testset.graph import KnowledgeGraph

QUESTIONS_PER_COUNTRY = 10

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
ragas_llm = llm_factory(
    model=os.environ["RAGAS_LLM_MODEL"],
    client=openrouter_client,
)
ragas_embeddings = embedding_factory(
    "openai",
    model=os.environ["EMBEDDING_MODEL"],
    client=openrouter_client,
)

all_samples: list[dict] = []

for country_code, docs in sorted(corpus_by_country.items()):
    print(f"Generating {QUESTIONS_PER_COUNTRY} questions for {country_code} ({len(docs)} doc(s))...")
    kg = KnowledgeGraph()
    generator = TestsetGenerator(
        llm=ragas_llm,
        embedding_model=ragas_embeddings,
        knowledge_graph=kg,
    )
    testset = generator.generate_with_langchain_docs(
        documents=docs,
        testset_size=QUESTIONS_PER_COUNTRY,
        raise_exceptions=False,
    )
    for row in testset.to_list():
        all_samples.append({
            "country_code": country_code,
            "question": row.get("user_input"),
            "reference": row.get("reference"),
        })
    print(f"  Done. {len(testset)} question(s) generated for {country_code}.")

print(f"\nTotal questions generated: {len(all_samples)}")

## Step 4 — Save dataset

In [ ]:
eval_dir = Path.cwd().parent / "eval"
eval_dir.mkdir(exist_ok=True)

output_path = eval_dir / "dataset.json"
output_path.write_text(
    json.dumps(all_samples, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

from collections import Counter
counts = Counter(s["country_code"] for s in all_samples)
print("Questions saved per country:")
for country_code, count in sorted(counts.items()):
    print(f"  {country_code}: {count}")
print(f"Total: {len(all_samples)} question(s) saved to {output_path}")